In [ ]:
#returns
import pandas as pd

def process_returns(input_file='returns.csv', output_file='returns.csv'):
    # Đọc dữ liệu thô
    df = pd.read_csv(input_file)

    df_clean = pd.DataFrame()

    # 1. Return_ID
    if 'return_id' in df.columns:
        df_clean['Return_ID'] = df['return_id'].astype(str).str.strip()
    elif 'Return_ID' in df.columns:
        df_clean['Return_ID'] = df['Return_ID'].astype(str).str.strip()
    else:
        df_clean['Return_ID'] = ['RET-' + str(i + 1).zfill(6) for i in range(len(df))]

    # 2. Return_Date
    date_col = 'return_date' if 'return_date' in df.columns else 'Return_Date'
    if date_col in df.columns:
        df_clean['Return_Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    else:
        df_clean['Return_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

    # 3. Reason
    reason_col = 'return_reason' if 'return_reason' in df.columns else 'Reason'
    if reason_col in df.columns:
        df_clean['Reason'] = df[reason_col].astype(str).str.strip().str.replace('_', ' ').str.title()
    else:
        df_clean['Reason'] = 'Unknown Reason'

    # 4. Return_Status (Nếu chưa có thì tự động gán REFUNDED cho các đơn đã hoàn tiền)
    if 'return_status' in df.columns:
        df_clean['Return_Status'] = df['return_status'].astype(str).str.strip().str.upper()
    elif 'Return_Status' in df.columns:
        df_clean['Return_Status'] = df['Return_Status'].astype(str).str.strip().str.upper()
    elif 'refund_amount' in df.columns:
        df_clean['Return_Status'] = df['refund_amount'].apply(lambda x: 'REFUNDED' if x > 0 else 'PENDING')
    else:
        df_clean['Return_Status'] = 'COMPLETED'

    # 5. Order_ID
    order_col = 'order_id' if 'order_id' in df.columns else 'Order_ID'
    df_clean['Order_ID'] = df[order_col].astype(str).str.strip() if order_col in df.columns else 'UNKNOWN'

    # 6. Product_ID
    product_col = 'product_id' if 'product_id' in df.columns else 'Product_ID'
    df_clean['Product_ID'] = df[product_col].astype(str).str.strip() if product_col in df.columns else 'UNKNOWN'

    # Sắp xếp đúng cấu trúc 6 cột theo yêu cầu
    cols = ['Return_ID', 'Return_Date', 'Reason', 'Return_Status', 'Order_ID', 'Product_ID']
    returns_table = df_clean[cols].drop_duplicates(subset=['Return_ID']).dropna(subset=['Return_ID'])

    # Xuất file CSV
    returns_table.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng RETURNS: {output_file} ({len(returns_table):,} dòng)")

if __name__ == '__main__':
    process_returns()

In [ ]:
import pandas as pd

def process_return_reasons(input_file='returns.csv', output_file='return_reasons.csv'):
    df = pd.read_csv(input_file)

    reason_col = 'return_reason' if 'return_reason' in df.columns else 'Reason'

    # Trích xuất lý do duy nhất
    reasons_df = pd.DataFrame(df[reason_col].unique(), columns=['Reason_Code'])
    reasons_df['Reason_Code'] = reasons_df['Reason_Code'].astype(str).str.strip().str.lower()
    reasons_df = reasons_df.drop_duplicates().reset_index(drop=True)

    # Tạo mã danh mục và tên hiển thị
    reasons_df['Reason_ID'] = ['REASON_' + str(i + 1).zfill(3) for i in range(len(reasons_df))]
    reasons_df['Reason_Name'] = reasons_df['Reason_Code'].str.replace('_', ' ').str.title()

    cols = ['Reason_ID', 'Reason_Code', 'Reason_Name']
    reasons_df = reasons_df[cols]

    reasons_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng RETURN_REASONS: {output_file} ({len(reasons_df):,} dòng)")

if __name__ == '__main__':
    process_return_reasons()

In [ ]:
import pandas as pd

def process_returns_summary(input_file='returns.csv', output_file='returns_summary.csv'):
    df = pd.read_csv(input_file)

    # Tự động nhận diện tên cột linh hoạt (viết hoa / viết thường)
    id_col = 'return_id' if 'return_id' in df.columns else ('Return_ID' if 'Return_ID' in df.columns else df.columns[0])
    reason_col = 'return_reason' if 'return_reason' in df.columns else ('Reason' if 'Reason' in df.columns else 'reason')
    qty_col = 'return_quantity' if 'return_quantity' in df.columns else 'Quantity'
    refund_col = 'refund_amount' if 'refund_amount' in df.columns else 'Refund_Amount'

    # Chuẩn hóa số liệu
    df['Quantity'] = pd.to_numeric(df[qty_col], errors='coerce').fillna(1).astype(int) if qty_col in df.columns else 1
    df['Refund'] = pd.to_numeric(df[refund_col], errors='coerce').fillna(0.0).round(2) if refund_col in df.columns else 0.0

    # Gom nhóm dữ liệu theo lý do trả hàng
    summary = df.groupby(reason_col).agg(
        Total_Return_Cases=(id_col, 'count'),
        Total_Quantity_Returned=('Quantity', 'sum'),
        Total_Refund_Amount=('Refund', 'sum')
    ).reset_index()

    summary.rename(columns={reason_col: 'Reason'}, inplace=True)
    summary['Reason'] = summary['Reason'].astype(str).str.replace('_', ' ').str.title()

    summary.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng RETURNS_SUMMARY: {output_file} ({len(summary):,} dòng)")

if __name__ == '__main__':
    process_returns_summary()